# Retail Supply Chain Analytics

## Predictive Analytics & Sales Forecasting

This notebook develops machine learning models to forecast weekly retail sales using the processed dataset created during the data preparation phase.

The workflow includes:

- Loading the processed dataset
- Feature selection
- Time-aware train/test splitting
- Model training
- Model evaluation
- Feature importance analysis
- Business interpretation

## Importing Libraries

The required Python libraries are imported for data manipulation, model development, evaluation and visualisation.

In [19]:
# =====================================================
# IMPORT LIBRARIES
# =====================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit

from sklearn.linear_model import LinearRegression

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## Loading the Processed Dataset

The modelling dataset generated during the feature engineering phase is loaded.

This dataset contains cleaned observations together with engineered calendar, sales and business features.

In [20]:
# =====================================================
# LOAD PROCESSED DATASET
# =====================================================

model_df = pd.read_csv(
    "../data/processed/model_dataset.csv"
)

print("Dataset loaded successfully.")

Dataset loaded successfully.


## Dataset Overview

Before modelling begins, the processed dataset is reviewed to confirm its dimensions, variables and data types.

In [21]:
# =====================================================
# DATASET OVERVIEW
# =====================================================

display(model_df.head())

print("Shape:", model_df.shape)

,store_id,department,date,weekly_sales,is_holiday,temperature,fuel_price,markdown_1,markdown_2,markdown_3,...,cumulative_sales,rolling_4_week_std,season_Spring,season_Summer,season_Winter,store_type_B,store_type_C,region_North,region_South,region_West
0,1,1,2022-01-08,35525.05,0,46.03,3.67,1356.75,2486.21,1427.01,...,154601.01,59079.415035,0,0,1,0,0,1,0,0
1,1,1,2022-01-15,14847.56,0,25.96,5.46,3861.22,596.15,22.09,...,169448.57,55184.346393,0,0,1,0,0,1,0,0
2,1,1,2022-01-22,45254.57,0,25.92,3.58,579.35,2589.31,2493.19,...,214703.14,45406.240356,0,0,1,0,0,1,0,0
3,1,1,2022-01-29,15166.69,0,78.37,4.41,4436.06,1416.64,478.38,...,229869.83,15184.020103,0,0,1,0,0,1,0,0
4,1,1,2022-02-05,15601.13,0,59.50,4.07,2137.71,76.26,431.57,...,245470.96,15027.895544,0,0,1,0,0,1,0,0


Shape: (155000, 32)


## Preparing the Date Column

The processed dataset was loaded from a CSV file, causing the `date` column to be stored as a string.

The column is converted back to the datetime format so that observations can be sorted chronologically and used for time-based model training and evaluation.

In [22]:
# =====================================================
# CONVERT DATE COLUMN TO DATETIME
# =====================================================

model_df["date"] = pd.to_datetime(model_df["date"])

print("Date converted successfully.")

print("\nDate Range:")
print("Start:", model_df["date"].min())
print("End  :", model_df["date"].max())

Date converted successfully.

Date Range:
Start: 2022-01-08 00:00:00
End  : 2024-12-21 00:00:00


## Sorting the Dataset

Time-series forecasting requires observations to remain in chronological order.

The dataset is sorted by date, store and department before splitting into training and testing data.

In [23]:
# =====================================================
# SORT DATASET
# =====================================================

model_df = model_df.sort_values(
    by=["date", "store_id", "department"]
).reset_index(drop=True)

display(model_df.head())

,store_id,department,date,weekly_sales,is_holiday,temperature,fuel_price,markdown_1,markdown_2,markdown_3,...,cumulative_sales,rolling_4_week_std,season_Spring,season_Summer,season_Winter,store_type_B,store_type_C,region_North,region_South,region_West
0,1,1,2022-01-08,35525.05,0,46.03,3.67,1356.75,2486.21,1427.01,...,154601.01,59079.415035,0,0,1,0,0,1,0,0
1,1,2,2022-01-08,64660.95,0,46.03,3.67,1356.75,2486.21,1427.01,...,183768.80,38499.772205,0,0,1,0,0,1,0,0
2,1,3,2022-01-08,72545.36,0,46.03,3.67,1356.75,2486.21,1427.01,...,156915.24,8361.198276,0,0,1,0,0,1,0,0
3,1,4,2022-01-08,80907.98,0,46.03,3.67,1356.75,2486.21,1427.01,...,169353.22,5329.647658,0,0,1,0,0,1,0,0
4,1,5,2022-01-08,79834.36,0,46.03,3.67,1356.75,2486.21,1427.01,...,144994.21,10376.445532,0,0,1,0,0,1,0,0


In [24]:
# =====================================================
# DATA TYPES
# =====================================================

model_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 155000 entries, 0 to 154999
Data columns (total 32 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   store_id                 155000 non-null  int64         
 1   department               155000 non-null  int64         
 2   date                     155000 non-null  datetime64[us]
 3   weekly_sales             155000 non-null  float64       
 4   is_holiday               155000 non-null  int64         
 5   temperature              155000 non-null  float64       
 6   fuel_price               155000 non-null  float64       
 7   markdown_1               155000 non-null  float64       
 8   markdown_2               155000 non-null  float64       
 9   markdown_3               155000 non-null  float64       
 10  markdown_4               155000 non-null  float64       
 11  markdown_5               155000 non-null  float64       
 12  cpi                      15

In [25]:
# =====================================================
# CHECK MISSING VALUES
# =====================================================

model_df.isnull().sum().sort_values(ascending=False)

store_id                   0
department                 0
date                       0
weekly_sales               0
is_holiday                 0
temperature                0
fuel_price                 0
markdown_1                 0
markdown_2                 0
markdown_3                 0
markdown_4                 0
markdown_5                 0
cpi                        0
unemployment               0
store_size                 0
year                       0
month                      0
quarter                    0
week_number                0
previous_week_sales        0
rolling_4_week_avg         0
weekly_sales_growth_pct    0
cumulative_sales           0
rolling_4_week_std         0
season_Spring              0
season_Summer              0
season_Winter              0
store_type_B               0
store_type_C               0
region_North               0
region_South               0
region_West                0
dtype: int64

## Defining the Target Variable

The objective of the forecasting models is to predict weekly sales.

The `weekly_sales` column is therefore selected as the target variable, while all remaining predictive variables are used as model features.

In [26]:
# =====================================================
# DEFINE TARGET VARIABLE
# =====================================================

target = "weekly_sales"

print(f"Target Variable: {target}")

Target Variable: weekly_sales


## Selecting Predictor Variables

The target variable (`weekly_sales`) is separated from the predictor variables.

Features that would introduce target leakage or are used only for chronological ordering are excluded from the modelling dataset.

In [27]:
# =====================================================
# DEFINE FEATURES AND TARGET
# =====================================================

target = "weekly_sales"

excluded_features = [
    "weekly_sales",
    "date",
    "cumulative_sales",
    "weekly_sales_growth_pct"
]

feature_columns = [
    column
    for column in model_df.columns
    if column not in excluded_features
]

X = model_df[feature_columns]
y = model_df[target]

print(f"Number of Features: {X.shape[1]}")
print(f"Number of Observations: {X.shape[0]}")

print("\nSelected Features:")
print(feature_columns)

Number of Features: 28
Number of Observations: 155000

Selected Features:
['store_id', 'department', 'is_holiday', 'temperature', 'fuel_price', 'markdown_1', 'markdown_2', 'markdown_3', 'markdown_4', 'markdown_5', 'cpi', 'unemployment', 'store_size', 'year', 'month', 'quarter', 'week_number', 'previous_week_sales', 'rolling_4_week_avg', 'rolling_4_week_std', 'season_Spring', 'season_Summer', 'season_Winter', 'store_type_B', 'store_type_C', 'region_North', 'region_South', 'region_West']


## Time-Based Train/Test Split

Unlike traditional machine learning problems, forecasting models must respect the chronological order of observations.

Instead of randomly splitting the dataset, the earliest observations are used for training while the most recent observations are reserved for testing.

This approach prevents data leakage and provides a realistic estimate of forecasting performance.

In [28]:
# =====================================================
# CHECK AVAILABLE DATE RANGE
# =====================================================

print("Start Date :", model_df["date"].min())
print("End Date   :", model_df["date"].max())

Start Date : 2022-01-08 00:00:00
End Date   : 2024-12-21 00:00:00


In [29]:
# =====================================================
# DETERMINE SPLIT DATE
# =====================================================

split_index = int(len(model_df) * 0.80)

split_date = model_df.iloc[split_index]["date"]

print("Training ends at :", split_date)

Training ends at : 2024-05-25 00:00:00


### Creating Training and Testing Datasets

The processed data is divided chronologically into training and testing sets.

The training set contains approximately 80% of the earliest observations and is used for model development.

The remaining 20% represents unseen future observations and is reserved for evaluating forecasting performance.

In [30]:
# =====================================================
# CREATE TRAIN AND TEST SETS
# =====================================================

train = model_df[model_df["date"] < split_date]

test = model_df[model_df["date"] >= split_date]

print("Training Shape :", train.shape)
print("Testing Shape  :", test.shape)

print("\nTraining Period")
print(train["date"].min(), "to", train["date"].max())

print("\nTesting Period")
print(test["date"].min(), "to", test["date"].max())

Training Shape : (124000, 32)
Testing Shape  : (31000, 32)

Training Period
2022-01-08 00:00:00 to 2024-05-18 00:00:00

Testing Period
2024-05-25 00:00:00 to 2024-12-21 00:00:00


### Preparing Model Inputs

The predictor variables and target variable are separated for both the training and testing datasets.

This creates the input matrices (`X`) and target vectors (`y`) required for model training.

In [31]:
# =====================================================
# SPLIT FEATURES AND TARGET
# =====================================================

X_train = train[feature_columns]
X_test = test[feature_columns]

y_train = train[target]
y_test = test[target]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (124000, 28)
X_test : (31000, 28)
y_train: (124000,)
y_test : (31000,)


# Baseline Forecast

Before training machine learning models, it is important to establish a baseline.

A baseline represents a simple forecasting strategy that machine learning models should outperform.

In retail forecasting, a common baseline is to predict that next week's sales will be the same as the previous week's sales.

The baseline provides a reference point for evaluating whether more sophisticated models offer meaningful improvements.

In [32]:
# =====================================================
# BASELINE MODEL
# Predict next week's sales using previous week's sales
# =====================================================

baseline_predictions = X_test["previous_week_sales"]

print("Baseline predictions created successfully.")

display(
    pd.DataFrame({
        "Actual Sales": y_test.head(10),
        "Baseline Prediction": baseline_predictions.head(10)
    })
)

Baseline predictions created successfully.


,Actual Sales,Baseline Prediction
124000,95076.55,69236.25
124001,9039.96,72249.79
124002,9186.00,33044.71
124003,89478.69,28717.40
124004,63905.84,41359.60
124005,118439.95,103452.74
124006,107848.50,40762.64
124007,76043.15,101407.22
124008,75654.38,127830.43
124009,124547.60,66505.20


## Evaluating the Baseline Model

The baseline model is evaluated using three standard regression metrics:

- **Mean Absolute Error (MAE):** Average prediction error in the same units as sales.
- **Root Mean Squared Error (RMSE):** Penalises larger prediction errors more heavily than MAE.
- **R² Score:** Measures how much of the variation in weekly sales is explained by the model.

These baseline metrics provide a benchmark against which machine learning models will be compared.

In [33]:
# =====================================================
# IMPORT EVALUATION METRICS
# =====================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [34]:
# =====================================================
# EVALUATE BASELINE MODEL
# =====================================================

import numpy as np

baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)

baseline_mse = mean_squared_error(
    y_test,
    baseline_predictions
)

baseline_rmse = np.sqrt(baseline_mse)

baseline_r2 = r2_score(
    y_test,
    baseline_predictions
)

print(f"Baseline MAE : {baseline_mae:,.2f}")
print(f"Baseline RMSE: {baseline_rmse:,.2f}")
print(f"Baseline R²  : {baseline_r2:.4f}")

Baseline MAE : 34,984.77
Baseline RMSE: 51,029.25
Baseline R²  : -0.0226


# Linear Regression Model

Linear Regression is one of the most widely used supervised learning algorithms for regression problems.

It models the relationship between the predictor variables and weekly sales by fitting a linear equation that minimises the prediction error on the training data.

Although retail sales are often influenced by complex non-linear relationships, Linear Regression provides an important benchmark against which more advanced machine learning models can be compared.

In [35]:
# =====================================================
# IMPORT LINEAR REGRESSION
# =====================================================

from sklearn.linear_model import LinearRegression

In [36]:
# =====================================================
# TRAIN LINEAR REGRESSION MODEL
# =====================================================

linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

print("Linear Regression model trained successfully.")

Linear Regression model trained successfully.


## Making Predictions

After training, the Linear Regression model is used to predict weekly sales for the unseen testing dataset.

These predictions will be compared with the actual sales values using the same evaluation metrics applied to the baseline model.

In [37]:
# =====================================================
# GENERATE PREDICTIONS
# =====================================================

linear_predictions = linear_model.predict(X_test)

print("Predictions generated successfully.")

display(
    pd.DataFrame({
        "Actual Sales": y_test.head(10).values,
        "Predicted Sales": linear_predictions[:10]
    })
)

Predictions generated successfully.


,Actual Sales,Predicted Sales
0,95076.55,73560.455505
1,9039.96,51471.534720
2,9186.00,43475.642209
3,89478.69,100867.012904
4,63905.84,91248.541957
5,118439.95,89323.175687
6,107848.50,67215.001519
7,76043.15,76542.515824
8,75654.38,59834.490774
9,124547.60,65312.613825


## Evaluating the Linear Regression Model

The Linear Regression model is evaluated using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and the coefficient of determination (R²).

The results will be compared directly with the baseline model to determine whether the model has learned meaningful relationships from the engineered features.

In [38]:
# =====================================================
# EVALUATE LINEAR REGRESSION MODEL
# =====================================================

linear_mae = mean_absolute_error(
    y_test,
    linear_predictions
)

linear_mse = mean_squared_error(
    y_test,
    linear_predictions
)

linear_rmse = np.sqrt(linear_mse)

linear_r2 = r2_score(
    y_test,
    linear_predictions
)

print(f"Linear Regression MAE : {linear_mae:,.2f}")
print(f"Linear Regression RMSE: {linear_rmse:,.2f}")
print(f"Linear Regression R²  : {linear_r2:.4f}")

Linear Regression MAE : 20,673.14
Linear Regression RMSE: 29,584.10
Linear Regression R²  : 0.6563


## Baseline vs Linear Regression

The evaluation metrics of the baseline model and the Linear Regression model are compared to determine whether the additional predictor variables improve forecasting performance.

A lower MAE and RMSE indicate smaller prediction errors, while a higher R² indicates that the model explains more variation in weekly sales.

In [39]:
# =====================================================
# COMPARE BASELINE AND LINEAR REGRESSION
# =====================================================

comparison = pd.DataFrame({
    "Model": ["Baseline", "Linear Regression"],
    "MAE": [baseline_mae, linear_mae],
    "RMSE": [baseline_rmse, linear_rmse],
    "R²": [baseline_r2, linear_r2]
})

comparison

,Model,MAE,RMSE,R²
0,Baseline,34984.769257,51029.254608,-0.022644
1,Linear Regression,20673.141182,29584.096186,0.656282


# Random Forest Regression

Random Forest is an ensemble machine learning algorithm that combines the predictions of many decision trees.

Unlike Linear Regression, Random Forest can model non-linear relationships and complex interactions between variables without requiring extensive feature transformations.

It is particularly well suited for retail sales forecasting because sales are influenced by factors such as promotions, holidays, seasonality, store characteristics, and customer behaviour that rarely follow simple linear patterns.

In [40]:
# =====================================================
# IMPORT RANDOM FOREST
# =====================================================

from sklearn.ensemble import RandomForestRegressor

## Training the Random Forest Model

The model is initially trained using a moderate number of trees.

This provides strong predictive performance while keeping training time manageable. Hyperparameter tuning can be performed later to further improve accuracy.

In [41]:
# =====================================================
# TRAIN RANDOM FOREST MODEL
# =====================================================

random_forest = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

random_forest.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


## Generating Predictions

After training, the Random Forest model predicts weekly sales for the testing dataset.

These predictions are then evaluated using the same metrics as the previous models to enable a fair comparison.

In [42]:
# =====================================================
# RANDOM FOREST PREDICTIONS
# =====================================================

rf_predictions = random_forest.predict(X_test)

display(
    pd.DataFrame({
        "Actual Sales": y_test.head(10).values,
        "Predicted Sales": rf_predictions[:10]
    })
)

,Actual Sales,Predicted Sales
0,95076.55,69150.8087
1,9039.96,47318.1600
2,9186.00,39315.2259
3,89478.69,101433.6094
4,63905.84,87634.3006
5,118439.95,89105.1501
6,107848.50,65555.8448
7,76043.15,75106.0281
8,75654.38,67877.3311
9,124547.60,60011.1081


## Evaluating the Random Forest Model

The predictive performance of the Random Forest model is assessed using MAE, RMSE, and R².

These metrics are compared against both the baseline and Linear Regression models to determine whether the ensemble approach provides a meaningful improvement.

In [43]:
# =====================================================
# RANDOM FOREST EVALUATION
# =====================================================

rf_mae = mean_absolute_error(
    y_test,
    rf_predictions
)

rf_mse = mean_squared_error(
    y_test,
    rf_predictions
)

rf_rmse = np.sqrt(rf_mse)

rf_r2 = r2_score(
    y_test,
    rf_predictions
)

print(f"Random Forest MAE : {rf_mae:,.2f}")
print(f"Random Forest RMSE: {rf_rmse:,.2f}")
print(f"Random Forest R²  : {rf_r2:.4f}")

Random Forest MAE : 18,805.02
Random Forest RMSE: 27,516.27
Random Forest R²  : 0.7027


## Model Comparison

The three models developed so far are compared using identical evaluation metrics.

This comparison highlights the improvement gained by progressively moving from a naïve forecasting strategy to increasingly sophisticated machine learning algorithms.

In [44]:
# =====================================================
# MODEL COMPARISON
# =====================================================

comparison = pd.DataFrame({
    "Model": [
        "Baseline",
        "Linear Regression",
        "Random Forest"
    ],
    "MAE": [
        baseline_mae,
        linear_mae,
        rf_mae
    ],
    "RMSE": [
        baseline_rmse,
        linear_rmse,
        rf_rmse
    ],
    "R²": [
        baseline_r2,
        linear_r2,
        rf_r2
    ]
})

comparison

,Model,MAE,RMSE,R²
0,Baseline,34984.769257,51029.254608,-0.022644
1,Linear Regression,20673.141182,29584.096186,0.656282
2,Random Forest,18805.023284,27516.271828,0.702652


# XGBoost Regression

Extreme Gradient Boosting (XGBoost) is an advanced ensemble learning algorithm based on gradient-boosted decision trees.

Unlike Random Forest, which builds many independent trees, XGBoost builds trees sequentially, with each new tree attempting to correct the errors made by the previous ones.

XGBoost is widely used in retail forecasting because it provides excellent predictive performance, handles nonlinear relationships effectively, and includes built-in regularisation to reduce overfitting.

In [45]:
# =====================================================
# IMPORT XGBOOST
# =====================================================

from xgboost import XGBRegressor

## Training the XGBoost Model

The model is initially trained using a balanced set of hyperparameters that provide good predictive performance while maintaining reasonable training time.

Hyperparameter tuning can be explored later to further improve accuracy.

In [46]:
# =====================================================
# TRAIN XGBOOST MODEL
# =====================================================

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)

xgb_model.fit(X_train, y_train)

print("XGBoost model trained successfully.")

XGBoost model trained successfully.


## Generating Predictions

The trained XGBoost model is used to predict weekly sales for the testing dataset.

These predictions are then compared with the observed sales values.

In [47]:
# =====================================================
# XGBOOST PREDICTIONS
# =====================================================

xgb_predictions = xgb_model.predict(X_test)

display(
    pd.DataFrame({
        "Actual Sales": y_test.head(10).values,
        "Predicted Sales": xgb_predictions[:10]
    })
)

,Actual Sales,Predicted Sales
0,95076.55,67165.875000
1,9039.96,46961.843750
2,9186.00,45633.128906
3,89478.69,99141.070312
4,63905.84,87660.398438
5,118439.95,87526.125000
6,107848.50,69435.554688
7,76043.15,77442.617188
8,75654.38,60483.800781
9,124547.60,67166.343750


## Evaluating the XGBoost Model

The predictive performance of the XGBoost model is evaluated using MAE, RMSE, and R².

These metrics are compared with the previously developed models to determine whether gradient boosting provides the highest forecasting accuracy.

In [49]:
# =====================================================
# XGBOOST EVALUATION
# =====================================================

xgb_mae = mean_absolute_error(
    y_test,
    xgb_predictions
)

xgb_mse = mean_squared_error(
    y_test,
    xgb_predictions
)

xgb_rmse = np.sqrt(xgb_mse)

xgb_r2 = r2_score(
    y_test,
    xgb_predictions
)

print(f"XGBoost MAE : {xgb_mae:,.2f}")
print(f"XGBoost RMSE: {xgb_rmse:,.2f}")
print(f"XGBoost R²  : {xgb_r2:.4f}")

XGBoost MAE : 18,658.26
XGBoost RMSE: 27,332.28
XGBoost R²  : 0.7066


## Final Model Comparison

The performance of all forecasting models is compared using identical evaluation metrics.

This comparison identifies the best-performing model for weekly retail sales forecasting.

In [50]:
# =====================================================
# FINAL MODEL COMPARISON
# =====================================================

comparison = pd.DataFrame({
    "Model": [
        "Baseline",
        "Linear Regression",
        "Random Forest",
        "XGBoost"
    ],
    "MAE": [
        baseline_mae,
        linear_mae,
        rf_mae,
        xgb_mae
    ],
    "RMSE": [
        baseline_rmse,
        linear_rmse,
        rf_rmse,
        xgb_rmse
    ],
    "R²": [
        baseline_r2,
        linear_r2,
        rf_r2,
        xgb_r2
    ]
})

comparison

,Model,MAE,RMSE,R²
0,Baseline,34984.769257,51029.254608,-0.022644
1,Linear Regression,20673.141182,29584.096186,0.656282
2,Random Forest,18805.023284,27516.271828,0.702652
3,XGBoost,18658.264650,27332.282351,0.706615
